<a href="https://colab.research.google.com/github/francielleneves/vitrinifarne-agente/blob/main/03_interface_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente Vitrinifarne — Notebook 3: interface web

Aqui a gente testa a aplicação de verdade — o `app.py` e o `src/agente.py` — antes de
subir para a Oracle Cloud.

O Gradio gera um link público temporário. Você abre esse link no navegador (ou manda para
alguém), conversa com o agente e tira o print para o README.

**Pré-requisito:** o Secret `GOOGLE_API_KEY` configurado, com "Acesso ao notebook" ligado.

---

## 1. Instalar

Mesma lista do `requirements.txt`, mais o Gradio. Cerca de dois minutos.

In [ ]:
!pip install -q gradio langchain-google-genai langchain-text-splitters faiss-cpu pypdf python-docx python-pptx openpyxl beautifulsoup4 pandas
print("Pronto.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.5 MB/s eta 0:00:00
Pronto.


## 2. Baixar o projeto e preparar o índice

Clonamos o repositório e chamamos `Agente().preparar()`. Na primeira execução ele lê os
8 documentos, gera os embeddings e **salva o índice na pasta `indice/`**.

Se você rodar esta célula uma segunda vez, ela carrega o índice do disco em menos de um
segundo, sem gastar cota de API. É exatamente esse comportamento que vai fazer o servidor
da OCI subir rápido.

In [ ]:
import os, sys
from pathlib import Path
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY").strip()

!rm -rf vitrinifarne-agente
!git clone -q https://github.com/francielleneves/vitrinifarne-agente.git

RAIZ = Path("vitrinifarne-agente").resolve()
sys.path.insert(0, str(RAIZ / "src"))

from agente import Agente

agente = Agente().preparar()
print("\nAgente pronto.")

8 documentos lidos.
64 pedaços gerados. Gerando embeddings...
  10/64
  20/64
  30/64
  40/64
  50/64
  60/64
  64/64
Índice criado: 64 vetores de 3072 dimensões.
Índice salvo em /content/vitrinifarne-agente/indice

Agente pronto.


## 3. Conferir antes de abrir a tela

Uma pergunta rápida, só para garantir que o modelo está respondendo. Se der erro aqui,
não vale abrir a interface — o problema é no agente, não no Gradio.

In [ ]:
print(agente.responder("Qual o prazo para desistir de uma compra?"))

O prazo para desistir de uma compra (arrependimento) é de 7 dias corridos a contar da data de recebimento do produto.

Fontes:
- Tabela 1
- Política de Reembolso e Devoluções — versão 2.2

---
Trechos consultados: Termos e Condições de Uso e de Compra (v1.4); Política de Reembolso e Devoluções (v2.2); Perguntas Frequentes (v3.0)


## 4. Subir a interface

O `share=True` cria um endereço público que funciona por 72 horas. Vão aparecer dois links:

- **local** — só funciona dentro do Colab, ignore
- **public** — termina em `.gradio.live`, é esse que você abre e compartilha

A célula fica rodando enquanto o servidor está ativo. Para encerrar, use o botão de parar
da célula.

Enquanto estiver no ar, faça as perguntas dos exemplos e **tire print da tela** —
principalmente de uma resposta com as fontes citadas. Esse print entra no README.

In [ ]:
import gradio as gr

EXEMPLOS = [
    "Qual o prazo para desistir de uma compra?",
    "Em quantas vezes posso parcelar?",
    "Quanto tempo demora para o dinheiro voltar?",
    "Um cliente quer devolver um produto de higiene pessoal com o lacre aberto. Pode?",
    "Comprei uma estante modular para Belém. Quando chega e o frete é grátis?",
    "Quais estados ficam na região Sul para efeito de frete?",
]

DESCRICAO = """Assistente interno da **Vitrinifarne**, loja online de casa e decoração.

Responde com base em 8 documentos oficiais da empresa, em 8 formatos diferentes.
Toda resposta cita o documento e a versão consultados. Quando a informação não está
na documentação, o assistente informa isso em vez de improvisar."""


def conversar(mensagem, historico):
    return agente.responder(mensagem)


gr.ChatInterface(
    fn=conversar,
    title="Agente Vitrinifarne",
    description=DESCRICAO,
    examples=EXEMPLOS,
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d34e0fe5a02254ddd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## Se algo der errado

**Erro no import do `agente`** — o `src/agente.py` não está no repositório, ou está em
outra pasta. Confira no GitHub.

**"Defina a variável de ambiente GOOGLE_API_KEY"** — o Secret não está com "Acesso ao
notebook" ligado para este notebook.

**O link `.gradio.live` não abre** — rode a célula 4 de novo. O túnel do Gradio cai às
vezes.

**A resposta vem sem as fontes** — o agente respondeu que não encontrou a informação.
Isso é comportamento correto, não erro.

## Depois deste notebook

Só falta a implantação na OCI e o README. O código da aplicação já é o definitivo — o que
roda aqui é o mesmo que vai rodar no servidor.